# 🍲 Nhận Diện Thức Ăn Bằng CNN (Dữ liệu thật)

Mô hình CNN nhận diện 11 loại món ăn. Khi upload ảnh, hệ thống sẽ dự đoán đó là món gì và in ra chi tiết món ăn, giá cả, và hàm lượng dinh dưỡng.

In [ ]:
!pip install tensorflow pandas numpy matplotlib opencv-python-headless scikit-learn ipywidgets

## 1. Kết nối Google Drive (Chỉ dùng cho Colab)

In [ ]:
# Chạy ô này để kết nối Google Drive (Chỉ có tác dụng trên Google Colab)
try:
    from google.colab import drive
    print("==================================================")
    print("BẠN MUỐN DÙNG TÀI KHOẢN GOOGLE NÀO?")
    print("[1] Giữ nguyên Nick Google hiện tại")
    print("[2] ĐỔI SANG NICK MỚI (Ép đăng nhập lại)")
    print("==================================================")
    
    choice = input("👉 Nhập số 1 hoặc 2 rồi bấm Enter: ")
    
    if choice.strip() == '2':
        print("\nĐang ép thoát nick cũ! Vui lòng làm theo hướng dẫn để đăng nhập bằng NICK MỚI...")
        drive.mount('/content/drive', force_remount=True)
    else:
        print("\nĐang kiểm tra kết nối với nick hiện tại...")
        drive.mount('/content/drive')
        
    print("✅ Đã kết nối với Google Drive thành công!")
except ImportError:
    print("ℹ️ BẠN ĐANG CHẠY TRÊN MÁY CÁ NHÂN (VS CODE). Bỏ qua bước này.")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory

print("Đã import đầy đủ thư viện!")

## 2. Thông Tin Món Ăn và Load Dữ Liệu

In [ ]:
# Thông tin chi tiết cho từng class món ăn (cập nhật 11 món)
FOOD_INFO = {
    'ca_hu_kho': {'name': 'Cá hú kho', 'price': '30,000 VNĐ', 'nutrition': '320 Kcal | 20g Protein | 22g Fat'},
    'canh_chua_co_ca': {'name': 'Canh chua có cá', 'price': '25,000 VNĐ', 'nutrition': '180 Kcal | 15g Protein | 8g Fat'},
    'canh_chua_khong_ca': {'name': 'Canh chua không cá', 'price': '10,000 VNĐ', 'nutrition': '90 Kcal | 3g Protein | 2g Fat'},
    'canh_rau': {'name': 'Canh rau', 'price': '8,000 VNĐ', 'nutrition': '45 Kcal | 2g Protein | 1g Fat'},
    'com_trang': {'name': 'Cơm trắng', 'price': '5,000 VNĐ', 'nutrition': '200 Kcal | 4g Protein | 45g Carbs'},
    'dau_hu_sot_ca': {'name': 'Đậu hũ sốt cà', 'price': '15,000 VNĐ', 'nutrition': '150 Kcal | 10g Protein | 9g Fat'},
    'rau_xao': {'name': 'Rau xào', 'price': '12,000 VNĐ', 'nutrition': '80 Kcal | 2g Protein | 6g Fat'},
    'suon_nuong': {'name': 'Sườn nướng', 'price': '35,000 VNĐ', 'nutrition': '380 Kcal | 28g Protein | 24g Fat'},
    'thit_kho': {'name': 'Thịt kho', 'price': '25,000 VNĐ', 'nutrition': '350 Kcal | 22g Protein | 25g Fat'},
    'thit_kho_trung': {'name': 'Thịt kho trứng', 'price': '30,000 VNĐ', 'nutrition': '400 Kcal | 25g Protein | 30g Fat'},
    'trung_chien': {'name': 'Trứng chiên', 'price': '10,000 VNĐ', 'nutrition': '120 Kcal | 10g Protein | 8g Fat'}
}

print("Đã định nghĩa thông tin dinh dưỡng và giá cả!")

In [ ]:
# Tự động phát hiện đường dẫn dù bạn chạy trên Colab hay máy tính (VS Code)
colab_path_1 = '/content/drive/MyDrive/train'
colab_path_2 = '/content/drive/MyDrive/Food_Datasets/train'

# Sử dụng đường dẫn tương đối để tránh lỗi (áp dụng khi tải thư mục train đặt chung chỗ với file .ipynb này)
local_path_1 = './train'
local_path_2 = 'train'

# Chế độ chẩn đoán lỗi
print(f"Thư mục hiện tại của Notebook: {os.getcwd()}")
print(f"Danh sách file/thư mục tại đây: {os.listdir('.')}")

if os.path.exists(colab_path_1):
    dataset_dir = colab_path_1
elif os.path.exists(colab_path_2):
    dataset_dir = colab_path_2
elif os.path.exists(local_path_1):
    dataset_dir = local_path_1
elif os.path.exists(local_path_2):
    dataset_dir = local_path_2
elif os.path.exists(r'C:\Users\MR ASUS\Downloads\Foot\train'):
    dataset_dir = r'C:\Users\MR ASUS\Downloads\Foot\train'
else:
    dataset_dir = None

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

if dataset_dir is None:
    print("\n❌ LỖI: Không tìm thấy thư mục chứa dữ liệu!")
    print("GỢI Ý: Vui lòng đảm bảo thư mục 'train' đã được giải nén và nằm cùng cấp với file Notebook này.")
else:
    print("\n✅ Đã tìm thấy thư mục dataset tại:", dataset_dir)
    
    # Load tập huấn luyện (80%)
    train_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    
    # Load tập kiểm tra (20%)
    val_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    
    # Lấy danh sách các nhãn (tên thư mục)
    CLASS_NAMES = train_dataset.class_names
    NUM_CLASSES = len(CLASS_NAMES)
    print("\nCác món ăn đã tìm thấy:", CLASS_NAMES)
    import tensorflow as tf
    AUTOTUNE = tf.data.AUTOTUNE
    train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
    val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
    print("Da bat bo nho dem (Cache) de tang toc len gap 10 lan!")


## 3. Huấn Luyện Mạng CNN

In [ ]:
if dataset_dir is not None:
    from tensorflow.keras.applications import MobileNetV2
    
    # Nâng cấp lên mô hình Transfer Learning (MobileNetV2) để nhận diện chính xác hơn
    base_model = MobileNetV2(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), include_top=False, weights="imagenet")
    base_model.trainable = False # KHÓA LẠI ĐỂ TRÁNH LỖI HỎNG TRÍ NHỚ CỦA AI
    
    model = models.Sequential([
        # MobileNetV2 yêu cầu dải màu từ -1 đến 1
        layers.Rescaling(1./127.5, offset=-1, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        # Thêm Data Augmentation
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])
    
    # Tốc độ học chuẩn
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    
    print("\nBắt đầu huấn luyện siêu mô hình MobileNetV2 (Ổn định)...")
    # Có Early Stopping để chống quá khớp (overfitting)
    callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    history = model.fit(train_dataset, validation_data=val_dataset, epochs=15, callbacks=[callback])
    
    model.save("food_model_real.keras")
    print("✅ Đã lưu mô hình!")
else:
    print("Không có dữ liệu để huấn luyện!")


## 4. Giao Diện Upload Ảnh và Dự Đoán

In [ ]:
def predict_food(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print("Lỗi: Không thể đọc ảnh.")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()
    
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_tensor = np.expand_dims(img_resized, axis=0)
    
    predictions = model.predict(img_tensor, verbose=0)
    class_idx = np.argmax(predictions[0])
    confidence = predictions[0][class_idx] * 100
    
    predicted_folder = CLASS_NAMES[class_idx]
    info = FOOD_INFO.get(predicted_folder, {'name': predicted_folder, 'price': 'Liên hệ', 'nutrition': 'Đang cập nhật'})
    
    print("\n" + "="*50)
    print(f"🎯 NHẬN DIỆN MÓN ĂN: {info['name']}")
    print(f"Độ tin cậy: {confidence:.2f}%")
    print("-"*50)
    print(f"💰 GIÁ TIỀN:    {info['price']}")
    print(f"⚡ DINH DƯỠNG:  {info['nutrition']}")
    print("="*50)

if dataset_dir is not None:
    # Giao diện Upload
    uploader = widgets.FileUpload(accept='image/*', multiple=False, description='Tải ảnh lên', button_style='success')
    out = widgets.Output()
    
    def on_upload(change):
        with out:
            clear_output()
            if not uploader.value:
                return
                
            if isinstance(uploader.value, dict):
                fname = list(uploader.value.keys())[0]
                content = uploader.value[fname]['content']
            else:
                content = uploader.value[0]['content']
                
            temp_path = 'temp_food_predict.jpg'
            with open(temp_path, 'wb') as f:
                f.write(content)
                
            predict_food(temp_path)
            uploader.value.clear() if hasattr(uploader.value, 'clear') else None
    
    uploader.observe(on_upload, names='value')
    display(widgets.VBox([
        widgets.HTML("<h3>📸 BẤM NÚT ĐỂ TẢI ẢNH MÓN ĂN CỦA BẠN LÊN:</h3>"),
        uploader,
        out
    ]))
else:
    print("Không thể hiện giao diện dự đoán vì chưa load được dữ liệu!")